# Подсчёт мелких и перекрытых фигур

Подключите данные соревнования и приватный `clevr_detail`. Включите GPU T4 ×2 и запустите Run All. Для оценки используется исходный val.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display
from scipy.special import expit
from sklearn.metrics import roc_auc_score

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert torch.cuda.device_count() == 2, "Выберите GPU T4 x2"
ROOT = Path(tempfile.mkdtemp(prefix="clevr_verify_"))
RESULT = ROOT / "verify_result"
RESULT.mkdir()
print("Рабочая папка", ROOT, flush=True)

def unpack(path, dest):
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not (dest / name).resolve().is_relative_to(dest.resolve()):
                raise ValueError(name)
        z.extractall(dest)

def find_data(parent):
    return [p.parent for p in parent.rglob("train.csv")
            if all((p.parent / n).is_file() for n in ("val.csv","test.csv","sample_submission.csv"))
            and all((p.parent / n).is_dir() for n in ("train","val","test"))]

locations = list(INPUT.rglob("detail_manifest.json"))
if not locations:
    archives = list(INPUT.rglob("clevr_detail.zip"))
    if len(archives) != 1:
        raise ValueError("Подключите один Private Dataset из clevr_detail.zip")
    unpack(archives[0], ROOT / "package")
    locations = [ROOT / "package/detail_manifest.json"]
if len(locations) != 1:
    raise ValueError("Подключено несколько версий весов")
package = locations[0].parent
for key, digest in json.loads(locations[0].read_text()).items():
    with (package / key).open("rb") as f:
        if hashlib.file_digest(f, "sha256").hexdigest() != digest:
            raise ValueError(key)
datasets = find_data(INPUT)
if not datasets:
    for j, path in enumerate(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(path) as z:
            names = {Path(k).name for k in z.namelist()}
        if {"train.csv","val.csv","test.csv","sample_submission.csv"} <= names:
            unpack(path, ROOT / "inputs" / str(j))
    datasets = find_data(ROOT / "inputs")
if len(datasets) != 1:
    raise ValueError("Подключите данные соревнования")
DATA = datasets[0]
env = dict(os.environ, CLEVR_DATA=str(DATA), CLEVR_PRIORS=str(package/"priors"),
           CLEVR_WEIGHTS=str(package/"weights_in"), PYTHONPATH=str(package),
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")

In [ ]:
assert all("T4" in torch.cuda.get_device_name(i) for i in range(2)), "Выберите GPU T4 ×2"
env.update(FRAGMENT_INPUT=str(package), OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", NUMEXPR_NUM_THREADS="2")

RESULT = ROOT / "detail"
RESULT.mkdir(exist_ok=True)
COUNTS = ("pyramid","attention","paired","stride4","maskset")
TABLES = ("union","r50_only","local","contour","geometry","siglip")
SEEDS = (421,853)
COUNT_EPOCHS = 24
TABLE_EPOCHS = 16
C = "scripts.research.detail_counts"
T = "scripts.research.detail_union"
E = "scripts.research.detail_eval"
FAILURES = []
from threading import Lock
FAILURE_LOCK = Lock()
def launch(gpu, module, *args):
    step = module.rsplit(".",1)[-1] + "_" + "_".join(map(str,args)).replace("--","")
    done = RESULT / f"{step}.done"
    if done.exists():
        return
    log = RESULT / f"{step}.log"
    cmd = [sys.executable, "-m", module, *map(str,args)]
    with log.open("a", buffering=1) as out, subprocess.Popen(cmd, cwd=ROOT, text=True,
            env={**env,"CUDA_VISIBLE_DEVICES":str(gpu)}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as job:
        for line in job.stdout:
            print(f"[GPU {gpu}] {line}", end="", flush=True)
            out.write(line)
        if job.wait():
            shutil.copy2(log, WORK/f"visible_{step}_failed.log")
            raise RuntimeError(f"{step} завершился с ошибкой. Лог сохранён в Output.")
    done.touch()

def parallel(jobs):
    with ThreadPoolExecutor(max_workers=2) as pool:
        results = [pool.submit(fn,i,*args) for i,(fn,args) in enumerate(jobs)]
        for result in results:
            result.result()

def monitor(job, stage):
    telemetry = RESULT / f"gpu_{stage}.csv"
    with telemetry.open("w",buffering=1) as out:
        query = ["nvidia-smi", "--query-gpu=timestamp,index,utilization.gpu,utilization.memory,memory.used,temperature.gpu",
                 "--format=csv", "-l", "2"]
        try:
            process = subprocess.Popen(query,stdout=out,stderr=subprocess.STDOUT)
        except FileNotFoundError:
            process = None
        try:
            return job()
        finally:
            if process is not None:
                process.terminate()
                process.wait()

def safe(gpu,module,*args):
    try:
        launch(gpu,module,*args)
        return True
    except Exception as exc:
        failure = dict(module=module, args=list(args), error=repr(exc))
        with FAILURE_LOCK:
            FAILURES.append(failure)
            (RESULT/"failures.json").write_text(json.dumps(FAILURES,indent=2))
        print("Ошибка конфигурации",failure,flush=True)
        return False

def pair(module, arguments):
    with ThreadPoolExecutor(2) as pool:
        pending=[pool.submit(safe,gpu,module,*args) for gpu,args in enumerate(arguments)]
        return [job.result() for job in pending]

def fit_counts():
    for arm in COUNTS:
        status=pair(C,[("fit",arm,s,"--epochs",COUNT_EPOCHS) for s in SEEDS])
        with ThreadPoolExecutor(2) as pool:
            jobs=[pool.submit(safe,gpu,C,"val",arm,s) for gpu,s in enumerate(SEEDS) if status[gpu]]
            for job in jobs:
                job.result()

def extract(stage,splits):
    good=True
    for split in splits:
        status=pair(T,[(stage,"--split",split,"--shard",gpu) for gpu in (0,1)])
        good=all(status) and good
    return good

def fit_tables():
    if extract("extract",("train","val")):
        for arm in TABLES[:-1]:
            pair(T,[("fit","--arm",arm,"--seed",s,"--epochs",TABLE_EPOCHS) for s in SEEDS])
    if extract("extract_siglip",("train","val")):
        pair(T,[("fit","--arm","siglip","--seed",s,"--epochs",TABLE_EPOCHS) for s in SEEDS])

## Подсчёт до поиска центров и видимые маски

Сравниваются пирамида без внимания, внимание по фильтру, парное обучение и шаг 4 вместо 2. Модель набора масок обучается с сопоставлением один-к-одному. У весов R50 общая точка старта.

In [ ]:
monitor(fit_counts,"counts")

## Слабые предложения и объединение объектов

Сравниваются переобученные union и R50 без thaw, локальные признаки, контурные предложения, геометрия одной или двух фигур и признаки SigLIP2 в локальном кропе.

In [ ]:
monitor(fit_tables,"tables")

## Метрики и спорные сцены

В архиве сохраняются AUC, инверсии и ошибки по каждому запуску, таблица четырёх спорных сцен, контроль естественных перекрытий и телеметрия GPU.

In [ ]:
evaluation_ok = safe(0,E)
report = json.loads((RESULT/"detail_metrics.json").read_text()) if evaluation_ok else {}
if report:
    display(pd.DataFrame(report["baselines"]).T[["auc","inversions","errors"]])
    runs = pd.DataFrame(report["experiments"]).T
    if len(runs):
        display(runs.sort_values("auc",ascending=False).drop(columns="auc_by_category"))
    parts=[]
    for name,levels in report["overlaps"].items():
        parts.extend(dict(run=name, threshold=q, **values) for q,values in levels.items())
    if parts:
        display(pd.DataFrame(parts))
    if report.get("exact_count_diagnostics"):
        display(pd.DataFrame(report["exact_count_diagnostics"]).T)
    print("Выбрано для test",report.get("winner"),flush=True)

In [ ]:
winner = report.get("winner")
def test_selected():
    if not winner:
        return
    arm,method=winner
    if arm in COUNTS:
        done=pair(C,[("test",arm,s) for s in SEEDS])
    else:
        stage="extract_siglip" if arm=="siglip" else "extract"
        done=extract(stage,("test",)) and pair(T,[("infer","--arm",arm,"--seed",s,"--split","test") for s in SEEDS])
    if done and all(done) and safe(0,E,"--export"):
        filename=f"{arm}_{method}.csv"
        shutil.copy2(RESULT/filename,WORK/filename)
        display(FileLink(filename))
monitor(test_selected,"test")
with zipfile.ZipFile(WORK/"detail_results.zip", "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in RESULT.rglob("*"):
        if path.is_file() and path.suffix in {".json",".npz",".pt",".log",".csv"} and "tables" not in path.parts and not path.name.startswith("table_"):
            z.write(path,path.relative_to(RESULT))
os.chdir(WORK)
display(FileLink("detail_results.zip"))